In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction==2026.11.14 --extra-index-url https://download.easydiffraction.org/pypi/simple/


# La₀.₅Ba₀.₅CoO₃ and Si on a McStas instrument

&nbsp; &nbsp; &nbsp; &nbsp; &nbsp; &nbsp; 

Fit two phases, La₀.₅Ba₀.₅CoO₃ and silicon, to a time-of-flight pattern simulated with McStas.

It builds the catalogue case `pd-neut-tof_lbco-si-mcstas` step by step: the same project the
command line fits and the app's Examples open.

## Import the library

In [2]:
import easydiffraction as edi

## Define the project

A new project is empty: it holds no structure and no experiment until you add them.

In [3]:
project = edi.Project(name='pd-neut-tof_lbco-si-mcstas')

## Define the structures

Load the two structures from EasyDiffraction's collection. Each CIF gives its space group, cell
and atom sites.

In [4]:
path = edi.download_data('pd-neut-tof_lbco-si-mcstas-lbco', destination='data')
project.structures.add_from_cif_path(path)
path = edi.download_data('pd-neut-tof_lbco-si-mcstas-si', destination='data')
project.structures.add_from_cif_path(path)
lbco = project.structures['lbco']
si = project.structures['si']

Data 'pd-neut-tof_lbco-si-mcstas-lbco' is at data/pd-neut-tof_lbco-si-mcstas-lbco.edi
Data 'pd-neut-tof_lbco-si-mcstas-si' is at data/pd-neut-tof_lbco-si-mcstas-si.edi


In [5]:
project.display.structure('lbco')

View(html=<1,970,868 characters>, height=660)

## Define the experiment

Create the experiment from the measured data in EasyDiffraction's collection, then describe the
instrument, the peak profile, the background and the structure it links.

In [6]:
data_path = edi.download_data('pd-neut-tof_lbco-si-mcstas-mcstas-data', destination='data')
project.experiments.add_from_data_path(
    name='mcstas',
    data_path=data_path,
    sample_form='powder',
    beam_mode='time-of-flight',
    radiation_probe='neutron',
    scattering_type='bragg',
)
experiment = project.experiments['mcstas']
experiment.neutron_scattering_length = 'sears1992'
experiment.peak.type = 'tof-jorgensen'
experiment.instrument.name = 'McStas'
experiment.peak.broad_gauss_sigma_0 = 50000
experiment.peak.broad_gauss_sigma_1 = -50000
experiment.peak.broad_gauss_sigma_2 = 20000
experiment.peak.rise_alpha_1 = 0.01
experiment.peak.decay_beta_0 = 0.006
experiment.peak.decay_beta_1 = 0.004
experiment.peak.cutoff_fwhm = 30
experiment.instrument.setup_twotheta_bank = 94.90931761529106
experiment.instrument.calib_d_to_tof_linear = 58724.76869981215
experiment.excluded_regions.create(start=0, end=40000)
experiment.excluded_regions.create(start=108000, end=200000)
experiment.linked_structures.create(structure_id='lbco', scale=4)
experiment.linked_structures.create(structure_id='si', scale=0.2)
experiment.background.create(position=45000, intensity=0.23)
experiment.background.create(position=50000, intensity=0.23)
experiment.background.create(position=55000, intensity=0.23)
experiment.background.create(position=65000, intensity=0.23)
experiment.background.create(position=70000, intensity=0.23)
experiment.background.create(position=75000, intensity=0.23)
experiment.background.create(position=80000, intensity=0.23)
experiment.background.create(position=85000, intensity=0.23)
experiment.background.create(position=90000, intensity=0.23)
experiment.background.create(position=95000, intensity=0.23)
experiment.background.create(position=100000, intensity=0.23)
experiment.background.create(position=105000, intensity=0.23)
experiment.background.create(position=110000, intensity=0.23)

Data 'pd-neut-tof_lbco-si-mcstas-mcstas-data' is at data/pd-neut-tof_lbco-si-mcstas-mcstas.xye


In [7]:
project.display.pattern('mcstas')

View(html=<211,465 characters>, height=480)

## Perform the analysis

The fit goes in stages, freeing one group of parameters at a time.

### Scales and cells

Both phases' scales and cells put the peaks in place and at the right height; the DIFC stays
fixed, as in diffraction-lib's tutorial.

In [8]:
experiment.linked_structures['lbco'].scale.free = True
experiment.linked_structures['si'].scale.free = True
lbco.cell.length_a.free = True
si.cell.length_a.free = True

In [9]:
project.display.parameters.free()

#,datablock,category,entry,parameter,value,s.u.,min,max,units
1,lbco,cell,,length_a,3.8909,0,0,99,Å
2,si,cell,,length_a,5.43,0,0,99,Å
3,mcstas,linked_structure,lbco,scale,4,0,-inf,inf,
4,mcstas,linked_structure,si,scale,0.2,0,-inf,inf,


In [10]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_lbco-si-mcstas, 633 pts fitted / 1000 loaded, 4 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       50.18       398.16
      1      0.566     23.12        84.52   78.8% ↓
  done · converged · 1 iters · reduced χ² 84.52 · Rwp 23.12 % · 0.61 s


View(html=<88,863 characters>, height=376)

In [11]:
project.display.pattern('mcstas')

View(html=<212,887 characters>, height=480)

### Background

In [12]:
for point in experiment.background:
    point.intensity.free = True

In [13]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_lbco-si-mcstas, 633 pts fitted / 1000 loaded, 17 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       23.12        86.31
      1      0.584     21.09        71.82   16.8% ↓
  done · converged · 1 iters · reduced χ² 71.82 · Rwp 21.09 % · 0.63 s


View(html=<106,440 characters>, height=376)

In [14]:
project.display.pattern('mcstas')

View(html=<221,922 characters>, height=480)

### Peak widths

In [15]:
experiment.peak.broad_gauss_sigma_0.free = True
experiment.peak.broad_gauss_sigma_1.free = True
experiment.peak.broad_gauss_sigma_2.free = True

In [16]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_lbco-si-mcstas, 633 pts fitted / 1000 loaded, 20 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       21.09        72.17
      1      0.616     21.02        71.67   0.7% ↓
      2      0.691     21.00        71.52   0.2% ↓
      3      0.766     20.95        71.22   0.4% ↓
      4      0.842     20.90        70.86   0.5% ↓
      5      0.917     20.88        70.73   0.2% ↓
      6      1.027     20.88        70.73   
      7      1.115     20.88        70.73   
  done · converged · 7 iters · reduced χ² 70.73 · Rwp 20.88 % · 1.16 s


View(html=<110,738 characters>, height=376)

In [17]:
project.display.pattern('mcstas')

View(html=<222,026 characters>, height=480)

### Rise and decay

In [18]:
experiment.peak.rise_alpha_1.free = True
experiment.peak.decay_beta_0.free = True
experiment.peak.decay_beta_1.free = True

In [19]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_lbco-si-mcstas, 633 pts fitted / 1000 loaded, 23 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       20.88        71.07
      1      0.117     20.86        70.91   0.2% ↓
  done · converged · 1 iters · reduced χ² 70.91 · Rwp 20.86 % · 0.13 s


View(html=<114,402 characters>, height=376)

In [20]:
project.display.pattern('mcstas')

View(html=<222,017 characters>, height=480)

### Displacements

Co's and O's; La's, Ba's and Si's stay fixed, as the case's project keeps them.

In [21]:
lbco.atom_sites['Co'].adp_iso.free = True
lbco.atom_sites['O'].adp_iso.free = True

In [22]:
project.analysis.fit()
project.display.fit.results()

edi fit: project pd-neut-tof_lbco-si-mcstas, 633 pts fitted / 1000 loaded, 25 free
   iter   time (s)   Rwp (%)  reduced χ²   change
                       20.86        71.15
      1      0.102      5.50         4.94   93.1% ↓
      2      0.113      5.38         4.74   4.0% ↓
      3      0.125      5.27         4.54   4.3% ↓
      4      0.136      5.13         4.30   5.2% ↓
      5      0.148      5.06         4.18   2.8% ↓
      6      0.160      5.04         4.16   0.6% ↓
      7      0.172      5.04         4.16   
      8      0.198      5.04         4.16   
  done · converged · 8 iters · reduced χ² 4.16 · Rwp 5.04 % · 0.21 s


View(html=<117,909 characters>, height=376)

In [23]:
project.display.pattern('mcstas')

View(html=<222,457 characters>, height=480)

In [24]:
project.display.fit.correlations()

,1,2,3,4,5
1 lbco.atom_site.Co.adp_iso,,,,,
2 mcstas.linked_structure.lbco.scale,0.84,,,,
3 mcstas.peak.broad_gauss_sigma_2,0.02,0.14,,,
4 mcstas.peak.broad_gauss_sigma_0,-0.03,0.08,0.91,,
5 mcstas.peak.broad_gauss_sigma_1,0.01,-0.12,-0.97,-0.97,


In [25]:
project.display.structure('lbco')

View(html=<1,978,811 characters>, height=660)

In [26]:
project.display.structure('si')

View(html=<163,422 characters>, height=660)

## Report

The HTML report draws the measured, calculated and difference patterns with the reflection
positions, and lists the structures, the experiments and the fit. Open it in a browser.

In [27]:
project.report.save_html('pd-neut-tof_lbco-si-mcstas.html')

Saving report to 'pd-neut-tof_lbco-si-mcstas.html'


## Save the project

In [28]:
project.save_as('pd-neut-tof_lbco-si-mcstas')

Saving project to 'pd-neut-tof_lbco-si-mcstas'
